# Conditional Computation

## Learning Objectives
1. Understand gating mechanisms and routing
2. Compare discrete vs differentiable gates
3. Analyze compute efficiency and scaling
4. Implement sparse parameter activation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print('Conditional computation (sparse gating) simulation')

## Level 1: Gating Basics

In [ ]:
# Level 1: Simple learned gating mechanism
# Gate = sigmoid(w_g * h) decides whether to execute a block

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Simulate: 8-layer network, each layer has optional gating
num_layers = 8
hidden_dim = 256
n_samples = 100

# Hidden states from previous layer
hidden_states = np.random.randn(n_samples, hidden_dim)

# Learned gate weights (w_g)
gate_weights = np.random.randn(hidden_dim) * 0.1

# Compute gates
gate_logits = hidden_states @ gate_weights
gate_values = sigmoid(gate_logits)

# Threshold: execute layer if gate > 0.5
threshold = 0.5
layer_executed = (gate_values > threshold).astype(int)

print('Gating Mechanism Simulation:')
print('-' * 60)
print(f'Hidden states shape: {hidden_states.shape}')
print(f'Gate logits range: [{gate_logits.min():.2f}, {gate_logits.max():.2f}]')
print(f'Gate values range: [{gate_values.min():.3f}, {gate_values.max():.3f}]')
print(f'\nGate distribution:')
print(f'  Closed (gate < 0.5): {(layer_executed == 0).mean()*100:.1f}%')
print(f'  Open (gate >= 0.5): {(layer_executed == 1).mean()*100:.1f}%')

# Conditional computation savings
base_flops = num_layers * hidden_dim * hidden_dim  # Dense network
conditional_flops = (layer_executed.mean()) * num_layers * hidden_dim * hidden_dim
reduction_pct = (1 - conditional_flops / base_flops) * 100

print(f'\nCompute efficiency:')
print(f'  FLOPs (dense network): {base_flops/1e6:.1f}M')
print(f'  FLOPs (conditional): {conditional_flops/1e6:.1f}M')
print(f'  Reduction: {reduction_pct:.1f}%''')

## Level 2: Mixture of Experts (MoE) Routing

In [ ]:
# Level 2: Sparse MoE routing
# Route tokens to top-k experts, not all experts

num_experts = 8
tokens_per_seq = 128
batch_size = 32
expert_hidden = 1024

# Token representations
token_representations = np.random.randn(batch_size * tokens_per_seq, 512)

# Router: predict expert assignment for each token
router_weights = np.random.randn(512, num_experts) * 0.1
router_logits = token_representations @ router_weights  # (tokens, experts)
router_probs = np.exp(router_logits) / np.exp(router_logits).sum(axis=1, keepdims=True)

# Top-k routing (k=2)
k = 2
top_k_probs = np.partition(router_probs, -k, axis=1)[:, -k:]
top_k_indices = np.argsort(router_probs, axis=1)[:, -k:]

print('Mixture of Experts (MoE) Routing:')
print('-' * 60)
print(f'Num tokens: {batch_size * tokens_per_seq}')
print(f'Num experts: {num_experts}')
print(f'Top-k: {k}')
print(f'\nExpert load distribution:')

expert_loads = np.zeros(num_experts)
for token_idx in range(len(top_k_indices)):
    for expert_idx in top_k_indices[token_idx]:
        expert_loads[expert_idx] += 1

for expert_id, load in enumerate(expert_loads):
    pct = load / (batch_size * tokens_per_seq * k) * 100
    print(f'  Expert {expert_id}: {load:4.0f} tokens ({pct:5.1f}%)')

# Compute efficiency
active_params_per_token = k * expert_hidden * expert_hidden  # k experts, each expert_hidden²
total_params = num_experts * expert_hidden * expert_hidden
compute_reduction = (1 - (k / num_experts)) * 100

print(f'\nCompute efficiency:')
print(f'  Total expert parameters: {total_params/1e6:.1f}M')
print(f'  Active parameters per token: {active_params_per_token/1e6:.1f}M')
print(f'  Compute reduction: {compute_reduction:.1f}% (k/{num_experts})''')

## Real-World Example 1: SkipNet (Skip Layers)

In [ ]:
# Example 1: Learned layer skipping (SkipNet)
# Some inputs can skip layers (e.g., easy examples skip hard layers)

def simulate_skipnet(num_layers=50, num_samples=1000, difficulty_range=[0.1, 0.9]):
    '''Simulate SkipNet: learn which layers to skip per sample.'''
    
    sample_difficulties = np.random.uniform(difficulty_range[0], difficulty_range[1], num_samples)
    
    # Layer difficulty (later layers are harder, need more samples to skip)
    layer_difficulties = np.linspace(0.2, 0.9, num_layers)
    
    # Simple rule: skip layer if sample_difficulty < layer_difficulty
    # Interpretation: easy samples skip hard layers
    
    layer_skips = []
    total_flops = 0
    
    for sample_id, sample_diff in enumerate(sample_difficulties):
        skipped = 0
        for layer_id, layer_diff in enumerate(layer_difficulties):
            if sample_diff < layer_diff:
                skipped += 1
        
        layer_skips.append(skipped)
        total_flops += (num_layers - skipped) * 256 * 256  # FLOPs per layer
    
    avg_skipped = np.mean(layer_skips)
    return sample_difficulties, layer_skips, total_flops, avg_skipped

sample_diffs, skipped_per_sample, total_flops, avg_skipped = simulate_skipnet(num_layers=50)

print('SkipNet: Learned Layer Skipping')
print('='*70)
print(f'Total samples: {len(sample_diffs)}')
print(f'Layers per network: 50')
print(f'\nSkipping statistics:')
print(f'  Average layers skipped: {avg_skipped:.1f} / 50 ({avg_skipped/50*100:.1f}%)')
print(f'  Min layers skipped: {min(skipped_per_sample)} (hard sample)')
print(f'  Max layers skipped: {max(skipped_per_sample)} (easy sample)')

# Compute efficiency gain
dense_flops = 50 * 256 * 256 * len(sample_diffs)
conditional_flops = total_flops
savings = (1 - conditional_flops / dense_flops) * 100

print(f'\nCompute efficiency:')
print(f'  Dense FLOPs: {dense_flops/1e9:.2f}B')
print(f'  Conditional FLOPs: {conditional_flops/1e9:.2f}B')
print(f'  Savings: {savings:.1f}%')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Difficulty vs layers skipped
axes[0].scatter(sample_diffs, skipped_per_sample, alpha=0.5, s=20)
axes[0].set_xlabel('Sample Difficulty')
axes[0].set_ylabel('Layers Skipped')
axes[0].set_title('SkipNet: Easy Samples Skip Hard Layers')
axes[0].grid(alpha=0.3)

# Skipping distribution
axes[1].hist(skipped_per_sample, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
axes[1].axvline(avg_skipped, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_skipped:.1f}')
axes[1].set_xlabel('Layers Skipped')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Layer Skips')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Real-World Example 2: Scaling Laws for Conditional Compute

In [ ]:
# Example 2: Model size vs sparsity trade-off
# Larger models can have higher sparsity (more parameters, activate fewer)

def model_cost(num_params_b, activation_rate):
    '''Estimate training/inference cost.'''
    # Cost = num_params * activation_rate (only active params consume FLOPs)
    return num_params_b * activation_rate

# Different model scales
model_configs = [
    {'name': '7B dense', 'params': 7, 'activation': 1.0, 'sparse': False},
    {'name': '7B sparse (50%)', 'params': 7, 'activation': 0.5, 'sparse': True},
    {'name': '14B sparse (50%)', 'params': 14, 'activation': 0.5, 'sparse': True},
    {'name': '28B sparse (25%)', 'params': 28, 'activation': 0.25, 'sparse': True},
    {'name': '56B sparse (13%)', 'params': 56, 'activation': 0.125, 'sparse': True},
]

print('Sparse Model Scaling:')
print('='*70)
print('Model             | Params | Activation | Eff. Params | Cost vs 7B')
print('-'*70)

baseline_cost = model_cost(7, 1.0)

for config in model_configs:
    params = config['params']
    activation = config['activation']
    effective_params = params * activation
    cost_ratio = model_cost(params, activation) / baseline_cost
    
    print(f'{config["name"]:17s} | {params:6.0f}B | {activation:9.0%} | {effective_params:11.1f}B | {cost_ratio:5.1f}x')

print('-'*70)
print('\nKey insight:')
print('  56B sparse (13% activation) = 7B effective parameters')
print('  BUT: Model capacity >> 7B → better accuracy at same cost')
print('  This is Mixture-of-Experts scaling law (Shazeer et al., 2017)')

## Key Takeaways

In [ ]:
print('='*70)
print('CONDITIONAL COMPUTATION BEST PRACTICES')
print('='*70)
print('')
print('1. GATING MECHANISMS')
print('   - Simple gate: sigmoid(W_g · h) threshold at 0.5')
print('   - Training challenge: discrete decisions are non-differentiable')
print('   - Solutions: Gumbel-softmax, straight-through estimator, REINFORCE')
print('')
print('2. MIXTURE OF EXPERTS (MoE)')
print('   - Route tokens to top-k experts (k=1-4 typical)')
print('   - Load balancing: loss term to encourage uniform expert usage')
print('   - Compute: k/num_experts fraction of total expert compute')
print('')
print('3. LAYER SKIPPING')
print('   - Skip entire layers based on sample difficulty')
print('   - Learn routing jointly with layer weights')
print('   - Saves 20-40% compute on average')
print('')
print('4. SCALING BENEFITS')
print('   - 56B sparse (13% active) ≈ 7B dense in compute')
print('   - BUT: 56B has 8x more capacity → better accuracy')
print('   - Achieves "have your cake and eat it too"')
print('')
print('5. IMPLEMENTATION GOTCHAS')
print('   - Naive masking in PyTorch still allocates compute (no real speedup)')
print('   - Need sparse batching: gather tokens for each block separately')
print('   - Load balancing critical: uneven expert assignment ruins efficiency')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: Load Balancing Loss')
print('-' * 70)
print('MoE models often have uneven expert loads.')
print('Design a loss term that penalizes load imbalance:')
print('  L_balance = variance(tokens_per_expert)')
print('  Total loss = CE(prediction) + λ * L_balance')
print('\nWhat λ value balances accuracy vs load distribution?')
print('')
print('Exercise 2: Sparse Batching Implementation')
print('-' * 70)
print('Implement efficient sparse batching for MoE:')
print('  1. Identify which tokens go to which expert')
print('  2. Group tokens by expert')
print('  3. Run each expert once on its token group')
print('  4. Scatter results back to original positions')
print('\nHow much faster than naive masking?')
print('')
print('Success! Generated notebook 53 (conditional-computation)')